In [ ]:
#Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here: Read the dataset Q1_data.csv using read_csv()
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:#Inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Write your code here:Display dataset information using info()
df.info()

In [ ]:
# Task 4: Write your code here: Show statistical description using describe()
df.describe()

In [ ]:
# Task 5: Write your code here:Plot the target distribution (delivery_time)
# target distribution (target variable)
plt.figure(figsize=(10, 5))
plt.plot(df['Delivery_Time'].dropna())
plt.title('target Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# target distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=2, edgecolor='black')
plt.title('target Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:Drop the 'Order_ID' column from the data
#delivery distance, restaurant preparation time, traffic conditions, and order details.
# Select relevant columns, we do not select 'Order_ID'
cols= ['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs','Delivery_Time']
df_clean = df[cols].copy()


In [ ]:
# Task 2: Write your code here:Handle missing values appropriately
# Do we have missing values?
def check_missing_values(df):
  # Get missing values using pandas
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")
check_missing_values(df)

In [ ]:
# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')
# Fill Courier_Experience_yrs  with mode - discrete feature, mode is most representative
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mean())
# Fill Delivery_Time with mode - discrete feature, mode is most representative
df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mean())

print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:Check and remove duplicates if any exist
#Check for duplicates
def check_duplicates(df):
#TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
check_duplicates(df)

In [ ]:
# Task 4: Write your code here:Encode categorical variables if needed
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_clean['Weather'] = le.fit_transform(df_clean['Weather'])
df_clean['Traffic_Level'] = le.fit_transform(df_clean['Traffic_Level'])
df_clean['Vehicle_Type'] = le.fit_transform(df_clean['Vehicle_Type'])
df_clean['Time_of_Day'] = le.fit_transform(df_clean['Time_of_Day'])

In [ ]:
# Task 4: Write your code here:Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import OneHotEncoder
# Encode categorical columns - converts text to integers  i try to make it with OneHotEncoder but i don't what is wrong
categorical_cols = pd.DataFrame({"Weather": ['Rainy', 'Snowy', 'Foggy', 'Clear', 'Windy'],"Traffic_Level":['Medium','Low','High'],"Time_of_Day":['Afternoon','Night','Evening'],"Vehicle_Type":['Scooter','Bike','Car']}) # DataFrame of one column
for col in categorical_cols:
    oh = OneHotEncoder(sparse_output=False)
    data_onehot_encoded = oh.fit_transform(col)

data_onehot_encoded.head()

In [ ]:
# Task 5: Write your code here:Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler #import StandardScaler
numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Delivery_Time")
scaler = StandardScaler()
# scale the `numerical_cols`
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
# Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  df[target_column].hist() # Yeah you can just do this :)
  plt.show()
check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:Split the dataset into features (X) and target (y)
# Define features and target
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
X = df_clean[cols].columns.drop("Delivery_Time")
y = df_clean['Delivery_Time']


In [ ]:
# Use previously generated random data (example: regression data)
from sklearn.model_selection import KFold

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)

# Train RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, max_depth=15,
class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
print("Model trained!")
# Predictions and metrics
y_pred = model.predict(X_test)

#Evaluate using MAE (Mean Absolute Error) ONLY
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE:  ${mae:,.2f}")

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Legendary']))

In [ ]:
# Task 1: Write your code here:
# Feature importance
importance = pd.DataFrame({
    'feature': cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_clean['Delivery_Time'].dropna(), bins=30, edgecolor='black', color='orange')
plt.title('Speed Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: